In [1]:
!pip install -q ultralytics pycocotools pandas scikit-learn
print('Done.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.0 MB/s eta 0:00:00
Done.


In [2]:
import os, glob
import numpy as np
from pathlib import Path

FOLD_ID = 1   
EPOCHS  = 50
BATCH   = 16
IMGSZ   = 512

DATASET_ROOT = Path('/kaggle/input/datasets/nttgaming/ripvis-cs431')
EXTRACT_DIR  = DATASET_ROOT / 'datasets'
WORK_DIR     = Path('/kaggle/working')
os.makedirs(WORK_DIR, exist_ok=True)

print(f'FOLD_ID : {FOLD_ID}')
print(f'Dataset : {EXTRACT_DIR}')
print(f'Work dir: {WORK_DIR}')
for sub in ['train/images', 'test/images',
            'train/train_with_additional_data.json', 'test/test_gt.json']:
    p = EXTRACT_DIR / sub
    print(f'  {sub}: {"OK" if p.exists() else "NOT FOUND"}')


FOLD_ID : 1
Dataset : /kaggle/input/datasets/nttgaming/ripvis-cs431/datasets
Work dir: /kaggle/working
  train/images: OK
  test/images: OK
  train/train_with_additional_data.json: OK
  test/test_gt.json: OK


In [3]:
import torch, torch.nn as nn
from ultralytics import YOLO
from ultralytics.nn.modules import CBAM, C2f

NECK_C2F_INDICES = {12, 15, 18, 21}

def add_cbam_hooks(yolo_model, cbam_state_dict=None):
    """
    Gan CBAM forward hook sau moi C2f trong Neck.
    - cbam_state_dict: neu truyen vao, load weights da train
                       neu None, khoi tao random (chi dung khi TRAIN)
    Returns: (cbam_modules, hooks)
    """
    seq = yolo_model.model.model
    cbam_modules = nn.ModuleDict()
    hooks = []

    for idx in sorted(NECK_C2F_INDICES):
        layer = seq[idx]
        if not isinstance(layer, C2f):
            print(f'  [WARN] Layer {idx} la {type(layer).__name__}, khong phai C2f — bo qua')
            continue
        c_out = layer.cv2.conv.out_channels
        key   = f'cbam_{idx}'
        cbam_modules[key] = CBAM(c_out)
        def make_hook(k):
            def hook(module, inp, output):
                return cbam_modules[k](output)
            return hook
        hooks.append(layer.register_forward_hook(make_hook(key)))
        print(f'  Hook CBAM(c={c_out}) tai layer {idx}')

    if cbam_state_dict is not None:
        cbam_modules.load_state_dict(cbam_state_dict)
        print(f'  Loaded CBAM weights from state_dict')
    else:
        print(f'  CBAM weights: random init (training mode)')

    yolo_model.model.cbam_modules = cbam_modules
    print(f'  Tong CBAM hooks: {len(hooks)}')
    return cbam_modules, hooks


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
from sklearn.model_selection import KFold

train_img_dir = EXTRACT_DIR / 'train' / 'images'
all_files = sorted(glob.glob(str(train_img_dir / '*.*')))
print(f'Total train images: {len(all_files)}')

kf = KFold(n_splits=5, shuffle=True, random_state=42)
splits = list(kf.split(all_files))

for fi, (tr_idx, va_idx) in enumerate(splits):
    tr_paths = [all_files[i] for i in tr_idx]
    va_paths = [all_files[i] for i in va_idx]
    (WORK_DIR / f'train_fold_{fi}.txt').write_text('\n'.join(tr_paths))
    (WORK_DIR / f'val_fold_{fi}.txt').write_text('\n'.join(va_paths))
    data_yaml = (
        f'path: {EXTRACT_DIR}\n'
        f'train: {WORK_DIR / f"train_fold_{fi}.txt"}\n'
        f'val: {WORK_DIR / f"val_fold_{fi}.txt"}\n'
        f'names:\n  0: rip_current\n'
    )
    (WORK_DIR / f'data_fold_{fi}.yaml').write_text(data_yaml)
    print(f'  Fold {fi}: train={len(tr_paths)}, val={len(va_paths)}')

print(f'\nSe train FOLD {FOLD_ID}.')


Total train images: 18389
  Fold 0: train=14711, val=3678
  Fold 1: train=14711, val=3678
  Fold 2: train=14711, val=3678
  Fold 3: train=14711, val=3678
  Fold 4: train=14712, val=3677

Se train FOLD 1.


In [5]:
import gc

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
print(f'Training FOLD {FOLD_ID} ...')

model = YOLO('yolov8n-seg.pt')
cbam_modules, hooks = add_cbam_hooks(model)  # random init cho training

model.train(
    data     = str(WORK_DIR / f'data_fold_{FOLD_ID}.yaml'),
    epochs   = EPOCHS,
    imgsz    = IMGSZ,
    batch    = BATCH,
    patience = 5,
    project  = str(WORK_DIR / 'yolo_rip_current'),
    name     = f'fold_{FOLD_ID}',
    device   = DEVICE,
    workers  = 2,    
    exist_ok = True,
    verbose  = True,
)

# Luu CBAM weights rieng (best.pt khong chua hook weights)
cbam_save_path = WORK_DIR / 'yolo_rip_current' / f'fold_{FOLD_ID}' / 'weights' / 'cbam.pt'
torch.save(cbam_modules.state_dict(), cbam_save_path)
print(f'CBAM weights saved: {cbam_save_path}')

for h in hooks: h.remove()
del model, cbam_modules
gc.collect(); torch.cuda.empty_cache()
print(f'FOLD {FOLD_ID} done!')


GPU: Tesla T4
Training FOLD 1 ...
  Hook CBAM(c=128) tai layer 12
  Hook CBAM(c=64) tai layer 15
  Hook CBAM(c=128) tai layer 18
  Hook CBAM(c=256) tai layer 21
  CBAM weights: random init (training mode)
  Tong CBAM hooks: 4
Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_fold_1.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0

In [6]:
import json, cv2
from pycocotools.coco import COCO
import pycocotools.mask as maskUtils

IOU_THRESHOLDS = np.arange(0.40, 1.00, 0.05).tolist()

def compute_composite_score(gt_json_path, pred_json_path, valid_img_names=None):
    zero = {'F1[50]':0., 'F2[50]':0., 'F1[40:95]':0., 'F2[40:95]':0., 'Score':0.}
    if not os.path.exists(str(pred_json_path)) or not os.path.exists(str(gt_json_path)):
        return zero
    pred_data = json.load(open(str(pred_json_path)))
    if not pred_data:
        return zero
    gt_coco   = COCO(str(gt_json_path))
    pred_coco = gt_coco.loadRes(pred_data)
    fname_to_id = {i['file_name']: i['id'] for i in gt_coco.dataset['images']}
    valid_ids = (
        {fname_to_id[n] for n in valid_img_names if n in fname_to_id}
        if valid_img_names else set(gt_coco.getImgIds())
    )
    counts = [[0, 0, 0] for _ in IOU_THRESHOLDS]
    for img_id in valid_ids:
        gt_anns = [a for a in gt_coco.loadAnns(gt_coco.getAnnIds(imgIds=img_id)) if 'segmentation' in a]
        pr_anns = sorted(
            [a for a in pred_coco.loadAnns(pred_coco.getAnnIds(imgIds=img_id)) if 'segmentation' in a],
            key=lambda x: x.get('score', 0), reverse=True
        )
        if not gt_anns and not pr_anns:
            continue
        gt_rle = [gt_coco.annToRLE(a) for a in gt_anns]
        pr_rle = [gt_coco.annToRLE(a) for a in pr_anns]
        if not pr_anns:
            for idx in range(len(IOU_THRESHOLDS)): counts[idx][2] += len(gt_anns)
            continue
        if not gt_anns:
            for idx in range(len(IOU_THRESHOLDS)): counts[idx][1] += len(pr_anns)
            continue
        ious = maskUtils.iou(pr_rle, gt_rle, np.zeros(len(gt_rle)))
        for idx, thr in enumerate(IOU_THRESHOLDS):
            matched_gt = np.zeros(len(gt_anns), dtype=bool)
            tp = fp = 0
            for i in range(len(pr_anns)):
                eligible = (ious[i] >= thr) & (~matched_gt)
                if eligible.any():
                    tp += 1
                    matched_gt[np.argmax(np.where(eligible, ious[i], -1.0))] = True
                else:
                    fp += 1
            counts[idx][0] += tp
            counts[idx][1] += fp
            counts[idx][2] += int((~matched_gt).sum())
    f1_list, f2_list = [], []
    for tp, fp, fn in counts:
        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_list.append(2 * p * r / (p + r) if (p + r) > 0 else 0)
        f2_list.append(5 * p * r / (4 * p + r) if (4 * p + r) > 0 else 0)
    return {
        'F1[50]'   : float(f1_list[2]),
        'F2[50]'   : float(f2_list[2]),
        'F1[40:95]': float(np.mean(f1_list)),
        'F2[40:95]': float(np.mean(f2_list)),
        'Score'    : float(0.25*f1_list[2] + 0.25*f2_list[2] + 0.25*np.mean(f1_list) + 0.25*np.mean(f2_list)),
    }

def predict_to_coco_json_yolo(model, img_paths, gt_json_path, out_json_path, device):
    if not os.path.exists(str(gt_json_path)):
        return None
    gt_coco      = COCO(str(gt_json_path))
    fname_to_id  = {i['file_name']: i['id'] for i in gt_coco.dataset['images']}
    img_info_map = {i['id']: i for i in gt_coco.dataset['images']}
    results_list = []
    speed_info   = None
    for img_path in img_paths:
        img_name = os.path.basename(img_path)
        if img_name not in fname_to_id:
            continue
        img_id  = fname_to_id[img_name]
        results = model.predict(source=str(img_path), device=device, verbose=False, imgsz=512, conf=0.25)
        result  = results[0]
        if speed_info is None:
            speed_info = result.speed
        if result.masks is not None:
            orig_h = img_info_map[img_id]['height']
            orig_w = img_info_map[img_id]['width']
            yolo_h, yolo_w = result.orig_shape
            scale_x = orig_w / yolo_w
            scale_y = orig_h / yolo_h
            for i, polygon in enumerate(result.masks.xy):
                if len(polygon) == 0:
                    continue
                scaled = polygon * np.array([scale_x, scale_y])
                blob   = np.zeros((orig_h, orig_w), dtype=np.uint8)
                cv2.fillPoly(blob, [scaled.astype(np.int32)], 1)
                if blob.sum() == 0:
                    continue
                rle = maskUtils.encode(np.asfortranarray(blob))
                rle['counts'] = rle['counts'].decode('utf-8')
                results_list.append({
                    'image_id'    : img_id,
                    'category_id' : 1,
                    'segmentation': rle,
                    'score'       : float(result.boxes.conf[i].item()),
                    'bbox'        : list(maskUtils.toBbox(rle)),
                    'area'        : int(maskUtils.area(rle)),
                })
    with open(str(out_json_path), 'w') as f:
        json.dump(results_list, f)
    return speed_info

print('Metric functions ready.')


Metric functions ready.


In [7]:
import gc, pandas as pd
from IPython.display import display

DEVICE = '0' if torch.cuda.is_available() else 'cpu'

gt_test_json  = EXTRACT_DIR / 'test'  / 'test_gt.json'
gt_train_json = EXTRACT_DIR / 'train' / 'train_with_additional_data.json'
test_img_dir  = EXTRACT_DIR / 'test'  / 'images'
test_files    = sorted(glob.glob(str(test_img_dir / '*.*')))

best_weights = WORK_DIR / 'yolo_rip_current' / f'fold_{FOLD_ID}' / 'weights' / 'best.pt'
val_txt      = WORK_DIR / f'val_fold_{FOLD_ID}.txt'

if not best_weights.exists():
    print(f'ERROR: best.pt not found at {best_weights}')
else:
    # Load fused model truc tiep, KHONG dung hook
    # best.pt da fused -> predict mask OK khong can CBAM hook
    model = YOLO(str(best_weights))

    model_layers = len(list(model.model.modules()))
    model_params = sum(x.numel() for x in model.model.parameters())

    # --- Val ---
    val_files     = val_txt.read_text().splitlines()
    val_img_names = [os.path.basename(p) for p in val_files]
    out_val       = WORK_DIR / f'pred_val_fold_{FOLD_ID}.json'
    print('Predicting val set...')
    predict_to_coco_json_yolo(model, val_files, gt_train_json, out_val, DEVICE)
    val_scores = compute_composite_score(gt_train_json, out_val, val_img_names)

    # --- Test ---
    out_test   = WORK_DIR / f'pred_test_fold_{FOLD_ID}.json'
    print('Predicting test set...')
    speed_info = predict_to_coco_json_yolo(model, test_files, gt_test_json, out_test, DEVICE)
    test_scores = compute_composite_score(gt_test_json, out_test)

    # --- Ket qua ---
    record = {'Fold': FOLD_ID}
    record.update({f'Fold_{k}': v for k, v in val_scores.items()})
    record.update({f'Test_{k}': v for k, v in test_scores.items()})
    df = pd.DataFrame([record])
    print(f'\n=== Fold {FOLD_ID} Results ===')
    display(df.round(6))

    csv_path = WORK_DIR / f'results_fold_{FOLD_ID}.csv'
    df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')

    if speed_info:
        wt = sum(speed_info.values())
        df_spd = pd.DataFrame([{
            'Fold'        : FOLD_ID,
            'Model'       : 'YOLOv8n-Seg+CBAM',
            'Layers'      : model_layers,
            'Parameters'  : f'{model_params:,}',
            'Preprocess'  : f'{speed_info.get("preprocess", 0):.2f}ms',
            'Inference'   : f'{speed_info.get("inference", 0):.2f}ms',
            'Postprocess' : f'{speed_info.get("postprocess", 0):.2f}ms',
            'Wall time'   : f'{wt:.2f}ms',
        }])
        print('\n=== Speed Profiling ===')
        display(df_spd)
        df_spd.to_csv(WORK_DIR / f'speed_fold_{FOLD_ID}.csv', index=False)

    del model
    gc.collect(); torch.cuda.empty_cache()
    print(f'\nFold {FOLD_ID} evaluation done.')


Predicting val set...
loading annotations into memory...
Done (t=0.81s)
creating index...
index created!
loading annotations into memory...
Done (t=0.55s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Predicting test set...
loading annotations into memory...
Done (t=0.30s)
creating index...
index created!
loading annotations into memory...
Done (t=0.13s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!

=== Fold 1 Results ===


,Fold,Fold_F1[50],Fold_F2[50],Fold_F1[40:95],Fold_F2[40:95],Fold_Score,Test_F1[50],Test_F2[50],Test_F1[40:95],Test_F2[40:95],Test_Score
0,1,0.912822,0.93519,0.647758,0.663631,0.78985,0.602729,0.546044,0.35539,0.321966,0.456532


Saved: /kaggle/working/results_fold_1.csv

=== Speed Profiling ===


,Fold,Model,Layers,Parameters,Preprocess,Inference,Postprocess,Wall time
0,1,YOLOv8n-Seg+CBAM,262,"3,263,811",2.13ms,9.90ms,1.99ms,14.01ms



Fold 1 evaluation done.


In [8]:
import shutil

weights_dir = WORK_DIR / 'yolo_rip_current' / f'fold_{FOLD_ID}' / 'weights'

if not (weights_dir / 'best.pt').exists():
    print(f'ERROR: best.pt not found')
else:
    bundle_dir = WORK_DIR / f'fold_{FOLD_ID}_output'
    bundle_dir.mkdir(exist_ok=True)

    # Copy weights (best + last + cbam)
    for fname in ['best.pt', 'last.pt', 'cbam.pt']:
        src = weights_dir / fname
        if src.exists():
            shutil.copy(src, bundle_dir / fname)

    # Copy CSVs
    for csv_name in [f'results_fold_{FOLD_ID}.csv', f'speed_fold_{FOLD_ID}.csv']:
        src = WORK_DIR / csv_name
        if src.exists():
            shutil.copy(src, bundle_dir / csv_name)

    # Zip
    zip_path = WORK_DIR / f'fold_{FOLD_ID}_output'
    shutil.make_archive(str(zip_path), 'zip', root_dir=str(bundle_dir))
    shutil.rmtree(bundle_dir)

    final_zip = WORK_DIR / f'fold_{FOLD_ID}_output.zip'
    size_mb   = final_zip.stat().st_size / 1e6
    print(f'Saved: {final_zip}  ({size_mb:.1f} MB)')
    print('=> Vao Output tab de download.')


Saved: /kaggle/working/fold_1_output.zip  (12.7 MB)
=> Vao Output tab de download.
